# LangChain L7 — Level 6 — Long-term memory
Rahul says "I prefer answers in bullet points" on Monday. On Tuesday, in a *new* conversation,
OpsPilot should still know that. Thread memory cannot help: it is a different thread.

```text
STATE              = what is happening right now (this run)
SHORT-TERM MEMORY  = this conversation           (thread checkpoint)
LONG-TERM MEMORY   = facts about a user/entity   (store, keyed by user id, across threads)
KNOWLEDGE          = external documents          (retrieval, next section)
```

LangGraph provides a **store**: a key-value memory organised by namespace, e.g.
`("users", "rahul")`. Tools reach it through `ToolRuntime`, which also carries the per-run
**context** (who is talking, what role they have). Note the discipline: the agent decides
*what* to remember; the application decides *where* it goes and *who* can read it.

### Step 1 — Context schema, store, and memory tools

`context_schema` declares what the application passes into each run (here: the user id).
Tools that declare a `runtime: ToolRuntime` parameter receive the store and the context; the
model never sees that parameter, so it cannot spoof a user id.

In [ ]:
from dataclasses import dataclass
from langchain.tools import ToolRuntime
from langgraph.store.memory import InMemoryStore

@dataclass
class Context:
    user_id: str = "anonymous"
    role: str = "support"          # used by the permission middleware in L10

@tool
def remember_preference(preference: str, runtime: ToolRuntime[Context]) -> str:
    """Save a lasting preference about how the current user wants to be helped."""
    namespace = ("preferences", runtime.context.user_id)
    existing = runtime.store.get(namespace, "list")
    items = (existing.value["items"] if existing else []) + [preference]
    runtime.store.put(namespace, "list", {"items": items})
    return f"Saved. {len(items)} preference(s) stored for {runtime.context.user_id}."

@tool
def recall_preferences(runtime: ToolRuntime[Context]) -> str:
    """Read the stored preferences of the current user."""
    existing = runtime.store.get(("preferences", runtime.context.user_id), "list")
    return json.dumps(existing.value["items"] if existing else [])

store = InMemoryStore()
opspilot_ltm = create_agent(
    model=model, tools=READ_TOOLS + [remember_preference, recall_preferences],
    system_prompt=OPSPILOT_PROMPT + " Before answering, recall the user's preferences if they ask how you should answer.",
    checkpointer=InMemorySaver(), store=store, context_schema=Context,
)
print("memory tools:", [t.name for t in (remember_preference, recall_preferences)])

### Step 2 — Remember in one thread, recall in another

Same user, two different conversations. The preference survives because it lives in the store
under the user's id, not in either thread's checkpoint. A different user sees nothing.

In [ ]:
monday = {"configurable": {"thread_id": "rahul-monday"}}
tuesday = {"configurable": {"thread_id": "rahul-tuesday"}}

out = opspilot_ltm.invoke({"messages": [{"role": "user", "content": "Please remember that I prefer short bullet-point answers."}]}, monday, context=Context(user_id="rahul"))
print("monday  :", text_of(out["messages"][-1])[:100])

out = opspilot_ltm.invoke({"messages": [{"role": "user", "content": "New conversation. What do you know about me and how should you answer?"}]}, tuesday, context=Context(user_id="rahul"))
print("tuesday :", text_of(out["messages"][-1])[:140])

out = opspilot_ltm.invoke({"messages": [{"role": "user", "content": "What do you know about me and how should you answer?"}]}, {"configurable": {"thread_id": "priya-1"}}, context=Context(user_id="priya"))
print("priya   :", text_of(out["messages"][-1])[:100])

print("\nstore contents:", [(item.namespace, item.value) for item in store.search(("preferences",))])

### Recap

- **Problem seen:** preferences vanished with the thread.
- **Layer added:** a store keyed by user id, `ToolRuntime` access from tools, and a `context_schema` set by the application.
- **Evidence:** Tuesday's new thread recalled Monday's preference; another user saw an empty list.